# ICMP Echo Request Payload Analysis and Enrichment

In [ ]:
import pandas as pd
import ipaddress
import os
import glob
import json

Output variables for input and output directories

In [ ]:
dir_input = "/Users/kix/src/ibram/08_ibrflow/test-nout4"
dir_output = "/Users/kix/src/ibram/08_ibrflow/test-nout4_enriched2"
file_mask = "darknet_*_mod_non_fragmented_1_echoscan-ipid_echoscan-noipid.csv"

## Enrichement functions for IBR ICMP stats dataset

These functions include some columns to the dataframe. These coloumns will be removed
and the final information is moved to the IBR info dictionary column.

In [ ]:
# This function converts the ip_src or ip_dst from ipaddress to hexadecimal
def ip_to_hex(ip_str):
    ip = ipaddress.ip_address(ip_str)
    return ip.packed.hex()

In [ ]:
def include_ip_hex_columns(df):
    df['ip_src_hex'] = df['ip_src'].apply(ip_to_hex)
    df['ip_dst_hex'] = df['ip_dst'].apply(ip_to_hex)
    return df

In [ ]:
def include_flag_address_in_payload_columns(df):
    # Create two boolean colums with checking the payload_hex includes the hex representation of the IP addresses
    df['ip_src_in_payload'] = df.apply(lambda row: row['ip_src_hex'] in row['payload_hex'], axis=1)
    df['ip_dst_in_payload'] = df.apply(lambda row: row['ip_dst_hex'] in row['payload_hex'], axis=1)
    return df

In [ ]:
def print_ip_address_in_payload_stats(filename, df):
    # Count the number of ocurrences where ip_src or ip_dst is found in the payload
    src_count = df['ip_src_in_payload'].sum()
    dst_count = df['ip_dst_in_payload'].sum()

    # Count the number of ocurrences where the ip_src and the ip_dst are both found in the payload
    both_count = df[(df['ip_src_in_payload']) & (df['ip_dst_in_payload'])].shape[0]

    print(f"Stats for file {filename}: Total: {df.shape[0]}. ip_src in payload: {src_count}. ip_dst in payload: {dst_count}. Both in payload: {both_count}.")

    # print(f"Number of payloads containing ip_src: {src_count}")
    # print(f"Number of payloads containing ip_dst: {dst_count}")
    # print(f"Number of payloads containing both ip_src and ip_dst: {both_count}")
    # print(f"Total number of payloads: {df.shape[0]}")

In [ ]:
def include_payload_length_column(df):
    # Include a new columen with the length of the payload in bytes
    df['payload_length_bytes'] = df['payload_hex'].apply(lambda x: len(x) // 2)
    return df

In [ ]:
def copy_info_to_ibrjson(df):
    # Copy the enrichment information to the IBR info dictionary column
    def update_ibr_info(row):
        ibr_info = row.get('ibr_info', {})
        ibr_info['ip_src_hex'] = row['ip_src_hex']
        ibr_info['ip_dst_hex'] = row['ip_dst_hex']
        ibr_info['ip_src_in_payload'] = row['ip_src_in_payload']
        ibr_info['ip_dst_in_payload'] = row['ip_dst_in_payload']
        ibr_info['payload_length_bytes'] = row['payload_length_bytes']
        return ibr_info

    df['ibr_info'] = df.apply(update_ibr_info, axis=1)
    return df

In [ ]:
def remove_temp_columns(df):
    # Remove the temporary columns used for enrichment
    df = df.drop(columns=['ip_src_hex', 'ip_dst_hex', 'ip_src_in_payload', 'ip_dst_in_payload', 'payload_length_bytes'])
    return df

## Load the dataset

Load the dataset using the icmp files form the specified directory.

In [ ]:
all_files = glob.glob(os.path.join(dir_input, file_mask))

In [ ]:
columns_out_icmp = [
    "timestamp", "ip_version", "ip_orig_tos", "ip_tos", "ip_prec", "ip_dscp",
    "ip_enc", "ip_len", "ip_id", "ip_ttl", "ip_chksum", "ip_src", "ip_dst",
    "ip_options", "icmp_type", "icmp_code", "icmp_id", "icmp_seq",
    "icmp_chksum", "payload_hex", "key", "IBR_info",
]

In [ ]:
# Function to create the output filename based on input filename and the path
def create_output_filename(input_filename, output_folder, suffix="_enriched"):
    basename = os.path.basename(input_filename)
    base, ext = os.path.splitext(basename)
    return os.path.join(output_folder, f"{base}{suffix}{ext}")

In [ ]:
# Create the output directory if it doesn't exist
os.makedirs(dir_output, exist_ok=True)

In [ ]:
# Use a loop to read, process, enrich and save each file
for file in all_files:
    print(f"Processing file: {file}")
    if file.endswith('.csv'):
        df = pd.read_csv(file, sep=';', names=columns_out_icmp)
    elif file.endswith('.csv.gz'):
        df = pd.read_csv(file, compression='gzip', sep=';', names=columns_out_icmp)
    else:
        print(f"Unsupported file format: {file}")
        continue

    # Further processing and enrichment steps would go here
    output_filename = create_output_filename(file, dir_output, suffix="_enriched")

    # Convert the IBR_info column from JSON string to actual dictionary
    df['IBR_info'] = df['IBR_info'].apply(json.loads)

    # Update the dataframe with enrichment functions
    df = include_ip_hex_columns(df)
    df = include_flag_address_in_payload_columns(df)
    df = include_payload_length_column(df)
    print_ip_address_in_payload_stats(file, df)
    df = copy_info_to_ibrjson(df)
    df = remove_temp_columns(df)

    # Save the enriched dataframe to a new CSV file
    print(f"Enriched data will be saved to: {output_filename}")
    df.to_csv(output_filename, sep=';', index=False)